In [16]:
import os
import time
import requests
from urllib.parse import quote_plus
from tqdm import tqdm
from dotenv import load_dotenv
import json
import pandas as pd

In [ ]:
# Load scopus API key from environment 
load_dotenv()
API_KEY = os.getenv("ELSEVIER_API_KEY") 

In [ ]:
API_KEY = "ENTER_YOUR_API_KEY_HERE"

In [ ]:
url = "https://api.elsevier.com/content/search/scopus?query=TITLE-ABS-KEY(test)&count=1"

headers = {"X-ELS-APIKey": API_KEY, "Accept": "application/json"}

response = requests.get(url, headers=headers)
print(response.status_code)
print(response.text)

In [ ]:
# Base endpoint for Scopus Search API
BASE_URL = "https://api.elsevier.com/content/search/scopus"

# -------------------------
# Query construction
# -------------------------
query_terms = [
    '("Fraud" OR "Fraudulent Activities" OR "Fraud Detection")',
    '("Crowdfunding" OR "Alternative Financ*")'
]

# Scope query to title, abstract, keywords
query_string = "TITLE-ABS-KEY(" + " AND ".join(query_terms) + ")"

# Date range: 1992–2024
query_string += " AND PUBYEAR > 1991 AND PUBYEAR < 2025"

# -------------------------

headers = {"X-ELS-APIKey": API_KEY, "Accept": "application/json"}

print(query_string)

In [ ]:
# TRIAL

params = {
    "query": query_string,
    "count": 10  # debugging: small number
}

response = requests.get(BASE_URL, headers=headers, params=params)
response.raise_for_status()
data = response.json()

import json
# Save full raw response for inspection
with open("reviews/scopus/scopus_debug_raw.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(response.text)

In [ ]:
# FULL SEARCH
count_per_request = 25 # max in paginated requests
start_index = 0
all_results = []

while True:
    params = {
        "query": query_string,
        "count": count_per_request,
        "start": start_index,
    }

    response = requests.get(BASE_URL, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()

    entries = data.get("search-results", {}).get("entry", [])
    if not entries:
        break

    all_results.extend(entries)

    # Progress info
    total_results = int(data["search-results"].get("opensearch:totalResults", 0))
    print(f"Fetched {len(entries)} records (start={start_index}) / Total: {total_results}")

    # Update start_index for next page
    start_index += count_per_request
    if start_index >= total_results:
        print("Reached total available records.")
        break

    # Safety delay to respect API rate limits
    time.sleep(0.5)

print(f"Done fetching. Total records retrieved: {len(all_results)}")

# -------------------------
# Save JSON for analysis
# -------------------------
with open("reviews/scopus/scopus_full_fraud_crowdfunding.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

In [ ]:
records = []

for entry in data:
    record = {}
    
    # Simple fields
    simple_fields = [
        "dc:title", "dc:creator", "prism:publicationName",
        "prism:eIssn", "prism:volume", "prism:issueIdentifier",
        "prism:pageRange", "prism:coverDate", "prism:coverDisplayDate",
        "prism:doi", "citedby-count", "prism:aggregationType",
        "subtype", "subtypeDescription", "source-id",
        "openaccess", "openaccessFlag", "dc:identifier", "eid", "prism:url"
    ]
    for field in simple_fields:
        record[field] = entry.get(field, "missing")
    
    # Affiliation: concatenate all affiliations as "affilname (city, country)"
    affiliations = entry.get("affiliation", [])
    if affiliations:
        aff_list = []
        for aff in affiliations:
            name = aff.get("affilname", "missing")
            city = aff.get("affiliation-city", "missing")
            country = aff.get("affiliation-country", "missing")
            aff_list.append(f"{name} ({city}, {country})")
        record["affiliation"] = "; ".join(aff_list)
    else:
        record["affiliation"] = "missing"
    
    # freetoreadLabel: join all values
    freetoread = entry.get("freetoreadLabel", {}).get("value", [])
    if freetoread:
        record["freetoreadLabel"] = "; ".join([v.get("$", "missing") for v in freetoread])
    else:
        record["freetoreadLabel"] = "missing"

    # Store the record
    records.append(record)

# Create DataFrame
df = pd.DataFrame(records)

# Fill any remaining NaNs with "missing"
df = df.fillna("missing")

df.head()

In [ ]:
records = []

for entry in data:
    record = {}
    
    # ---------------------
    # Basic fields
    # ---------------------
    simple_fields = [
        "dc:title", "dc:creator", "prism:publicationName",
        "prism:eIssn", "prism:issn", "prism:isbn",
        "prism:volume", "prism:issueIdentifier",
        "prism:pageRange", "prism:coverDate", "prism:coverDisplayDate",
        "prism:doi", "citedby-count", "prism:aggregationType",
        "subtype", "subtypeDescription", "source-id",
        "openaccess", "openaccessFlag", "dc:identifier", "eid", "prism:url",
        "authkeywords", "fund-sponsor", "fund-acr", "fund-no"
    ]
    for field in simple_fields:
        record[field] = entry.get(field, "missing")
    
    # ---------------------
    # Affiliation: flatten
    # ---------------------
    affiliations = entry.get("affiliation", [])
    if affiliations:
        aff_list = []
        for aff in affiliations:
            name = aff.get("affilname", "missing")
            city = aff.get("affiliation-city", "missing")
            country = aff.get("affiliation-country", "missing")
            aff_list.append(f"{name} ({city}, {country})")
        record["affiliation"] = "; ".join(aff_list)
        # Corresponding author country (first affiliation)
        record["corresponding_author_country"] = affiliations[0].get("affiliation-country", "missing")
    else:
        record["affiliation"] = "missing"
        record["corresponding_author_country"] = "missing"
    
    # ---------------------
    # FreetoreadLabel flatten
    # ---------------------
    freetoread = entry.get("freetoreadLabel", {}).get("value", [])
    if freetoread:
        record["freetoreadLabel"] = "; ".join([v.get("$", "missing") for v in freetoread])
    else:
        record["freetoreadLabel"] = "missing"
    
    # ---------------------
    # Links flatten
    # ---------------------
    links = entry.get("link", [])
    for link in links:
        ref = link.get("@ref", None)
        href = link.get("@href", "missing")
        if ref:
            record[f"link_{ref}"] = href
    
    records.append(record)

# ---------------------
# Create DataFrame
# ---------------------
df = pd.DataFrame(records)
df = df.fillna("missing")

print(df.shape)
df.head()

In [ ]:
# Save to CSV
df.to_csv("reviews/scopus/scopus_flattened.csv", index=False, encoding="utf-8")